In [0]:
# Cell 1: Konfigurasi dengan Secret Scope
from pyspark.sql.functions import *
import time

print("PostgreSQL Incremental Ingestion...")

# JDBC Configuration dengan Secret Scope
jdbc_url = "jdbc:postgresql://aws-1-ap-southeast-1.pooler.supabase.com:5432/wim"
jdbc_username = dbutils.secrets.get(scope="jdbc_supabase", key="username")
jdbc_password = dbutils.secrets.get(scope="jdbc_supabase", key="password")
configs = {
    "url": jdbc_url,
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "org.postgresql.Driver",
    "table": "customers"
}

# Target table
catalog = "main"
schema_bronze = "bronze"
bronze_table = f"{catalog}.{schema_bronze}.{configs['table']}"

print(f"Target: {bronze_table}")
print(f"Source: public.{configs['table']}")

In [0]:
# Cell 2: Incremental Load dengan Watermark
def jdbc_read_with_retry(url, query, user, password, driver, max_retries=3, retry_interval=30):
    """JDBC read dengan retry mechanism"""
    last_error = None
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[JDBC] Attempt {attempt}/{max_retries} - Connecting...")
            df = spark.read.format("jdbc") \
                .option("url", url) \
                .option("dbtable", query) \
                .option("user", user) \
                .option("password", password) \
                .option("driver", driver) \
                .option("fetchsize", 1000) \
                .load()
            
            row_count = df.count()
            print(f"[JDBC] SUCCESS - Fetched {row_count} rows")
            return df
            
        except Exception as e:
            last_error = e
            print(f"[JDBC] FAILED - Attempt {attempt}: {str(e)}")
            
            if attempt < max_retries:
                print(f"[JDBC] Retrying in {retry_interval}s...")
                time.sleep(retry_interval)
            else:
                raise Exception(f"All {max_retries} retries failed") from e
    
    raise Exception("Unexpected retry error") from last_error

# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_bronze}")

# Cek watermark terakhir
try:
    latest_ts = spark.sql(
        f"SELECT MAX(updated_at) AS max_date FROM {bronze_table}"
    ).collect()[0]["max_date"]
    print(f"Watermark terakhir: {latest_ts}")
except:
    latest_ts = None
    print("Full load - bronze table belum ada")

# Build query - PostgreSQL yang filter
if latest_ts is None:
    query = f"(SELECT * FROM public.{configs['table']} ORDER BY updated_at LIMIT 100) AS full_load"
    print("Mode: FULL LOAD (100 rows for testing)")
    print("PostgreSQL: SELECT * FROM public.customers LIMIT 100")
else:
    query = f"(SELECT * FROM public.{configs['table']} WHERE updated_at > '{latest_ts}' ORDER BY updated_at LIMIT 100) AS incremental"
    print(f"Mode: INCREMENTAL (after {latest_ts})")
    print(f"PostgreSQL: SELECT * WHERE updated_at > '{latest_ts}' LIMIT 100")
    print("Hanya data BARU yang dibaca dari source")

# Read dari PostgreSQL
print("\nMengirim query ke PostgreSQL...")
raw_df = jdbc_read_with_retry(
    url=configs["url"],
    query=query,
    user=configs["user"],
    password=configs["password"],
    driver=configs["driver"],
    max_retries=3,
    retry_interval=10
)

print(f"\nData diterima dari PostgreSQL: {raw_df.count()} rows")
print("Mekanisme: PostgreSQL filter by updated_at, kirim hanya data baru ke Databricks")

In [0]:
# Cell 3: Upsert dengan MERGE
from delta.tables import DeltaTable

# Tambah metadata
bronze_df = raw_df \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source", lit("postgresql.public." + configs["table"])) \
    .withColumn("_batch_id", concat(
        lit("batch_"),
        date_format(current_timestamp(), "yyyyMMdd_HHmmss"),
        lit("_"),
        expr("uuid()")
    ))

row_count = bronze_df.count()
print(f"Baris baru: {row_count}")

if row_count > 0:
    # Cek apakah bronze table sudah ada
    if spark.catalog.tableExists(bronze_table):
        print("Mode: UPSERT (MERGE)")
        
        # Load existing table
        target_table = DeltaTable.forName(spark, bronze_table)
        
        # MERGE: update jika customer_id sudah ada, insert jika baru
        target_table.alias("target").merge(
            bronze_df.alias("source"),
            "target.customer_id = source.customer_id"
        ).whenMatchedUpdate(
            condition="source.updated_at > target.updated_at",
            set={
                "name": "source.name",
                "email": "source.email",
                "city": "source.city",
                "status": "source.status",
                "created_at": "source.created_at",
                "updated_at": "source.updated_at",
                "_ingested_at": "source._ingested_at",
                "_source": "source._source",
                "_batch_id": "source._batch_id"
            }
        ).whenNotMatchedInsert(
            values={
                "customer_id": "source.customer_id",
                "name": "source.name",
                "email": "source.email",
                "city": "source.city",
                "status": "source.status",
                "created_at": "source.created_at",
                "updated_at": "source.updated_at",
                "_ingested_at": "source._ingested_at",
                "_source": "source._source",
                "_batch_id": "source._batch_id"
            }
        ).execute()
        
        print(f"MERGE completed - {row_count} records processed")
        
    else:
        # First run: create table dengan INSERT
        print("Mode: INITIAL INSERT")
        bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
        print(f"Table created dengan {row_count} records")
    
    # Verifikasi - cek tidak ada duplikasi
    print("\nVerifikasi duplikasi:")
    dup_check = spark.sql(f"""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT customer_id) as unique_customers
        FROM {bronze_table}
    """)
    display(dup_check)
    
    print("\nSample 5 records terbaru:")
    display(
        spark.sql(f"""
            SELECT * 
            FROM {bronze_table} 
            ORDER BY updated_at DESC 
            LIMIT 5
        """)
    )
else:
    print("Tidak ada data baru")

In [0]:
# Cell 4: (Optional) Optimize Table
# Jalankan setelah banyak incremental loads
spark.sql(f"OPTIMIZE {bronze_table} ZORDER BY (updated_at)")
print("Table optimized untuk query incremental lebih cepat")